In [0]:
#Voici le code pour lire les données de ADLS ( Azure Data Lake Storage ) 


storage_account_name = "name_of_your_storage_account"
container_name = "name_of_your_container"
storage_account_key = "key_of_your_storage_account" 
azure_key_conf = f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net"


# Définition du chemin d'accès vers la couche Bronze
bronze_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze/"

try:
    print("Début du chargement des fichiers Parquet depuis la couche bronze...\n")
    
    # 1. Chargement des Transactions
    df_transactions = (spark.read
                       .format("parquet")
                       .option(azure_key_conf, storage_account_key)
                       .load(f"{bronze_base_path}/transactions/"))
    print("✔ Source 'Transactions' chargée.")

    # 2. Chargement des Produits
    df_products = (spark.read
                   .format("parquet")
                   .option(azure_key_conf, storage_account_key)
                   .load(f"{bronze_base_path}/products/"))
    print("✔ Source 'Products' chargée.")

    # 3. Chargement des Magasins (Stores)
    df_stores = (spark.read
                 .format("parquet")
                 .option(azure_key_conf, storage_account_key)
                 .load(f"{bronze_base_path}/stores/"))
    print("✔ Source 'Stores' chargée.")

    # 4. Chargement des Clients (Customers issus de l'API JSON via ADF)
    df_customers = (spark.read
                    .format("parquet")
                    .option(azure_key_conf, storage_account_key)
                    .load(f"{bronze_base_path}/customers/"))
    print("✔ Source 'Customers' chargée.")
    
    print("\n[SUCCÈS] Les 4 DataFrames de la couche Bronze sont prêts à être transformés !")
    
    # Petit affichage de contrôle pour vérifier qu'on a bien tout
    print("\n--- Structure du DataFrame Transactions ---")
    df_transactions.printSchema()
    
    print("--- Aperçu rapide des Clients ---")
    display(df_customers.limit(5))
    
except Exception as e:
    print("\n[ERREUR] Un des dossiers n'a pas pu être chargé. Vérifie le statut du pipeline ADF.")
    print("Détail de l'erreur :", e)


Début du chargement des fichiers Parquet depuis la couche bronze...

✔ Source 'Transactions' chargée.
✔ Source 'Products' chargée.
✔ Source 'Stores' chargée.
✔ Source 'Customers' chargée.

[SUCCÈS] Les 4 DataFrames de la couche Bronze sont prêts à être transformés !

--- Structure du DataFrame Transactions ---
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_date: date (nullable = true)

--- Aperçu rapide des Clients ---


transaction_id,customer_id,product_id,store_id,quantity,transaction_date
1,227,8,4,4,2025-03-31
2,205,3,4,5,2024-11-12
3,216,2,2,3,2025-05-01
4,220,8,1,1,2024-11-02
5,205,5,2,1,2025-03-17


In [0]:
from pyspark.sql.functions import col, to_date, current_timestamp

print(" Starting Silver Layer Processing (Writing directly to Azure)...")

# 1. Setup Azure Data Lake paths and keys


azure_key_conf = f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net"
bronze_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"

# New path for the Silver layer in Azure!
silver_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"

# 2. Read and clean the transactions table
df_transactions_silver = spark.read.format("parquet").option(azure_key_conf, storage_account_key).load(f"{bronze_base_path}/transactions/") \
    .dropDuplicates() \
    .dropna(subset=["transaction_id", "customer_id"]) \
    .withColumn("transaction_date", to_date(col("transaction_date"))) \
    .withColumn("silver_ingestion_ts", current_timestamp())

# 3. Read and clean the customers table
df_customers_silver = spark.read.format("parquet").option(azure_key_conf, storage_account_key).load(f"{bronze_base_path}/customers/") \
    .dropDuplicates() \
    .dropna(subset=["customer_id"])

# 4. Read and clean the products table
df_products_silver = spark.read.format("parquet").option(azure_key_conf, storage_account_key).load(f"{bronze_base_path}/products/") \
    .dropDuplicates() \
    .dropna(subset=["product_id"])

# 5. Read and clean the stores table
df_stores_silver = spark.read.format("parquet").option(azure_key_conf, storage_account_key).load(f"{bronze_base_path}/stores/") \
    .dropDuplicates() \
    .dropna(subset=["store_id"])

print(" Data cleaning completed. Saving directly to Azure Data Lake (Silver Folder)...")

# 6. Direct (External) save to Azure
# We use option to pass the secret key during writing just as we did during reading
df_transactions_silver.write.format("parquet").mode("overwrite").option(azure_key_conf, storage_account_key).save(f"{silver_base_path}/transactions/")
df_customers_silver.write.format("parquet").mode("overwrite").option(azure_key_conf, storage_account_key).save(f"{silver_base_path}/customers/")
df_products_silver.write.format("parquet").mode("overwrite").option(azure_key_conf, storage_account_key).save(f"{silver_base_path}/products/")
df_stores_silver.write.format("parquet").mode("overwrite").option(azure_key_conf, storage_account_key).save(f"{silver_base_path}/stores/")

print(" Success! The Silver data is now physically in Azure!")

 Starting Silver Layer Processing (Writing directly to Azure)...
 Data cleaning completed. Saving directly to Azure Data Lake (Silver Folder)...
 Success! The Silver data is now physically in Azure!


In [0]:
from pyspark.sql.functions import sum, col, round

print(" Starting Gold Layer Processing (Reading from & Writing to Azure)...")

# 1. Setup base paths and keys
 

azure_key_conf = f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net"
silver_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"

# New path for the Gold layer in Azure!
gold_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/gold"

print(" Reading clean data from Azure Silver folder...")

# 2. Reading from the Silver folder in Azure using the key
df_trans = spark.read.format("parquet").option(azure_key_conf, storage_account_key).load(f"{silver_base_path}/transactions/")
df_prod = spark.read.format("parquet").option(azure_key_conf, storage_account_key).load(f"{silver_base_path}/products/")

# 3. Merge data and calculate revenue
df_enriched = df_trans.join(df_prod, "product_id", "left") \
    .withColumn("revenue", col("quantity") * col("price"))

# 4. Build the first table (Sales by Category - KPI)
df_gold_category_sales = df_enriched.groupBy("category") \
    .agg(
        round(sum("revenue"), 2).alias("total_sales"),
        sum("quantity").alias("total_units_sold")
    ).orderBy(col("total_sales").desc())

# 5. Build the second table (Comprehensive Analytics Table - One Big Table)
df_gold_full_analytics = df_enriched.select(
    col("transaction_id"),
    col("transaction_date"),
    col("product_name"),
    col("category"),
    col("quantity"),
    col("revenue")
)

print(" Saving Business KPIs directly to Azure Data Lake (Gold Folder)...")

# 6. Direct save to the Gold folder in Azure (in Parquet format)
df_gold_category_sales.write.format("parquet").mode("overwrite").option(azure_key_conf, storage_account_key).save(f"{gold_base_path}/category_sales_kpi/")
df_gold_full_analytics.write.format("parquet").mode("overwrite").option(azure_key_conf, storage_account_key).save(f"{gold_base_path}/full_sales_analytics/")

print(" Success! Your Data Lakehouse is complete. Go check the 'gold' folder in Azure!")

# Display the result to verify
display(df_gold_category_sales)

 Starting Gold Layer Processing (Reading from & Writing to Azure)...
 Reading clean data from Azure Silver folder...
 Saving Business KPIs directly to Azure Data Lake (Gold Folder)...
 Success! Your Data Lakehouse is complete. Go check the 'gold' folder in Azure!


category,total_sales,total_units_sold
Électronique,25380.82,41
Fitness,7162.00,28
Accessoires,2475.00,25
Papeterie,245.00,7


In [0]:
%sql
-- Simple financial query to find the top-selling categories
SELECT category, total_sales, total_units_sold 
FROM gold_category_sales
ORDER BY total_sales DESC;

category,total_sales,total_units_sold
Électronique,25380.82,41
Fitness,7162.00,28
Accessoires,2475.00,25
Papeterie,245.00,7


In [0]:
from pyspark.sql.functions import col

print("Preparing the comprehensive Gold Analytics Table...")

# 1. Read the cleaned tables from the Silver layer
df_trans = spark.read.table("silver_transactions")
df_prod = spark.read.table("silver_products")

# 2. Merge the data (Join) and select only the important columns for analysis
df_gold_full = df_trans.join(df_prod, "product_id", "left") \
    .withColumn("revenue", col("quantity") * col("price")) \
    .select(
        col("transaction_id"),
        col("transaction_date"),
        col("product_name"),
        col("category"),
        col("quantity"),
        col("revenue")
    )

# 3. Save the final table to the Metastore (error-resistant)
df_gold_full.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold_sales_analytics")

print(" Success! The 'gold_sales_analytics' table is ready.")

# 4. Display the final table
display(df_gold_full)

Preparing the comprehensive Gold Analytics Table...
 Success! The 'gold_sales_analytics' table is ready.


transaction_id,transaction_date,product_name,category,quantity,revenue
1,2025-03-31,Organiseur de Bureau,Accessoires,4,396.00
3,2025-05-01,Enceinte Bluetooth,Électronique,3,1198.50
13,2025-05-04,Tapis de Sport,Fitness,4,796.00
14,2024-07-17,Souris Sans Fil,Électronique,5,749.95
23,2025-04-30,Haltères,Fitness,2,1098.00
22,2024-11-16,Haltères,Fitness,4,2196.00
11,2024-08-11,Souris Sans Fil,Électronique,2,299.98
16,2024-11-29,Bouteille Isotherme,Fitness,4,356.00
28,2024-11-15,Montre Connectée,Électronique,3,3897.00
4,2024-11-02,Organiseur de Bureau,Accessoires,1,99.00


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.